<a href="https://colab.research.google.com/github/Janeth172/EDP1/blob/main/TAREA_SMA_WMA_EMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TAREA**

En este proyecto se desarrolla una herramienta interactiva en Python que permite consultar precios históricos de acciones y calcular tres indicadores utilizados en el análisis técnico: la **Media Móvil Simple (SMA)**, **la Media Móvil Ponderada (WMA)** y **la Media Móvil Exponencial (EMA)**.

---



**Importar librerías**

Para desarrollar el programa se utilizan diferentes bibliotecas especializadas en la descarga de información financiera, el procesamiento de datos, los cálculos matemáticos, la visualización de resultados y la creación de controles interactivos.

`yfinance`: Obtener información financiera de Yahoo Finance

`pandas`: Manipulación y análisis de datos estructurados

`numPy`: Operaciones matemáticas y cálculos numéricos

`matplotlib.pyplot`: Representaciones gráficas de los datos

`ipywidgets`: Crear elementos interactivos

In [35]:
# IMPORTAR LIBRERÍAS NECESARIAS
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# Mostrar tablas y otros elementos visuales
from IPython.display import display

# Descargar archivos generados
from google.colab import files

**Configuración de fechas y catálogo de acciones**

En esta sección se establece el intervalo de tiempo que se utilizará para obtener las cotizaciones históricas. También se crea un catálogo de empresas que el usuario podrá seleccionar mediante un menú desplegable

In [36]:
# 2. CONFIGURACIÓN DEL PERIODO DE ANÁLISIS
fecha_inicio = "2025-01-01"
fecha_fin = "2026-11-11"

# 3. CATÁLOGO DE ACCIONES DISPONIBLES
acciones = {
    "Google (Alphabet)": "GOOGL",
    "Microsoft": "MSFT",
    "Amazon": "AMZN",
    "Tesla": "TSLA",
    "Apple": "AAPL",
    "CEMEX México": "CEMEXCPO.MX"
}

**Calcular los indicadores**

Se define una función llamada `calcular_medias`, encargada de ejecutar el análisis financiero de la acción seleccionada.

La función recibe dos parámetros:
- accion: símbolo bursátil de la empresa que se desea analizar.
- n: número de periodos utilizados para calcular las medias móviles.

In [37]:
# 3. FUNCIÓN CALCULA INDICADORES
def calcular_medias(accion, n):

    # Descargar cotizaciones históricas
    datos_yahoo = yf.download(
        accion,
        start=fecha_inicio,
        end=fecha_fin,
        auto_adjust=False,
        progress=False
    )

    if datos_yahoo.empty:
        print("No se encontraron cotizaciones.")
        return

    # Extraer únicamente los cierres
    precios = datos_yahoo["Close"]

    # Algunas versiones devuelven una tabla
    if isinstance(precios, pd.DataFrame):
        precios = precios.iloc[:, 0]

    # Crear tabla de trabajo
    datos = pd.DataFrame({
        "Fecha": precios.index.strftime("%Y-%m-%d"),
        "Precio": precios.to_numpy(dtype=float)
    })

    if len(datos) < n:
        print("No hay suficientes datos para este periodo.")
        return

    # CÁLCULO DE LA MEDIA MÓVIL SIMPLE (SMA)
    datos["SMA"] = (
        datos["Precio"]
        .rolling(window=n, min_periods=n)
        .mean()
    )

    # CÁLCULO DE LA MEDIA MÓVIL PONDERADA (WMA)
    # Generamos una secuencia de pesos desde 1 hasta n
    pesos = np.arange(1, n + 1)

    datos["WMA"] = (
        datos["Precio"]
        .rolling(window=n, min_periods=n)
        .apply(
            lambda ventana: np.dot(ventana, pesos) / pesos.sum(),
            raw=True
        )
    )

    # CÁLCULO DE LA MEDIA MÓVIL EXPONENCIAL (EMA)
    # factor suavizamiento
    k = 2 / (n + 1)

    # Inicializar la columna EMA
    datos["EMA"] = np.nan

    # Primera EMA = SMA de los primeros n precios
    datos.loc[n - 1, "EMA"] = datos.loc[n - 1, "SMA"]

    # Calcular las EMA siguientes
    for i in range(n, len(datos)):
        precio_actual = datos.loc[i, "Precio"]
        ema_anterior = datos.loc[i - 1, "EMA"]

        datos.loc[i, "EMA"] = (
            precio_actual * k
            + ema_anterior * (1 - k)
        )

    # PRESENTACIÓN DE RESULTADOS
    print("Acción:", accion)
    print("Periodos:", n)
    print("Factor K:", k)

    display(datos.tail(10).round(4))

    # REPRESENTACIÓN GRÁFICA
    plt.figure(figsize=(13, 6))

     # GRÁFICA DE PRECIOS ORIGINALES
    plt.plot(
        datos["Fecha"], datos["Precio"],
        label="Precio de cierre", linewidth=1
    )

    # GRÁFICA DE LA SMA
    plt.plot(
        datos["Fecha"], datos["SMA"],
        label=f"SMA ({n})", linewidth=2
    )

    # GRÁFICA DE LA WMA
    plt.plot(
        datos["Fecha"], datos["WMA"],
        label=f"WMA ({n})", linewidth=2
    )

     # GRÁFICA DE LA EMA
    plt.plot(
        datos["Fecha"], datos["EMA"],
        label=f"EMA ({n})", linewidth=2
    )

    plt.title(f"Análisis técnico de {accion}")
    plt.xlabel("Fecha")
    plt.ylabel("Precio de la acción")
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Mostrar aproximadamente 8 fechas en el eje horizontal
    posiciones = np.linspace(
        0, len(datos) - 1,
        min(8, len(datos)),
        dtype=int
    )

    plt.xticks(
        posiciones,
        datos["Fecha"].iloc[posiciones],
        rotation=45
    )

    plt.tight_layout()
    plt.show()

    # EXPORTACIÓN DE RESULTADOS
    nombre_archivo = f"medias_{accion}_{n}.csv"

    datos.to_csv(
        nombre_archivo,
        index=False,
        float_format="%.12f"
    )

    print("Archivo generado:", nombre_archivo)
    files.download(nombre_archivo)


# CONSTRUCCIÓN DE CONTROLES INTERACTIVOS
# Creamos un menú desplegable para acciones
menu_acciones = widgets.Dropdown(
    options=acciones,
    description="Acción:"
)

# Creamos un menú desplegable para acciones
menu_periodos = widgets.IntSlider(
    value=20,
    min=2,
    max=100,
    step=1,
    description="Periodos:"
)

# EJECUCIÓN INTERACTIVA DEL PROGRAMA
interfaz = widgets.interact_manual(
    calcular_medias,
    accion=menu_acciones,
    n=menu_periodos
)

interfaz.widget.children[-2].description = "Calcular medias"


interactive(children=(Dropdown(description='Acción:', options={'Google (Alphabet)': 'GOOGL', 'Microsoft': 'MSF…